# Object Detection using YOLOv11

## Setup

In [ ]:
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import json
from pathlib import Path
from collections import defaultdict
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor


# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define the root path
RUN_ROOT = Path("/content/drive/My Drive/Deep_Learning_Project/Training_and_annotations")
TRAIN_JSON_PATH = RUN_ROOT / 'train.json'
TEST_JSON_PATH = RUN_ROOT / 'test.json'
# Update this if images are in a specific subfolder like RUN_ROOT / 'train_images'
IMAGE_DIR_TRAINING = RUN_ROOT / 'training'
IMAGE_DIR_TESTING = RUN_ROOT / 'testing'

# Determine device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
import json
from collections import defaultdict
from pathlib import Path
import matplotlib.patches as patches

# Assuming RUN_ROOT, TRAIN_JSON_PATH, IMAGE_DIR_TRAINING are defined from previous cells

# --- Load training data --- (redundant if mfdlJZkgyDKf has run, but good for self-containment)
cat_id_to_yolo = {}
yolo_names = {}
train_coco_data = None

if TRAIN_JSON_PATH.exists():
    with open(TRAIN_JSON_PATH, 'r') as f:
        train_coco_data = json.load(f)

    # Map category IDs to continuous YOLO indices (0, 1, 2...) from training data
    cats = sorted(train_coco_data['categories'], key=lambda x: x['id'])
    cat_id_to_yolo = {c['id']: i for i, c in enumerate(cats)}
    yolo_names = {i: c['name'] for i, c in enumerate(cats)}

    print(f"Classes mapping: {yolo_names}")
else:
    print(f"Error: Training JSON file not found at {TRAIN_JSON_PATH}.")

if train_coco_data:
    # Group image info by id
    img_id_to_info = {img['id']: img for img in train_coco_data['images']}

    # Group annotations by image_id
    img_to_anns = defaultdict(list)
    for ann in train_coco_data['annotations']:
        img_to_anns[ann['image_id']].append(ann)

    # --- Find one sample image for each class ---
    sample_images_per_class = {}
    classes_found = set()

    for img_info in train_coco_data['images']:
        img_id = img_info['id']
        if img_id not in img_to_anns: # Skip images with no annotations
            continue

        current_image_classes = set()
        for ann in img_to_anns[img_id]:
            coco_cat_id = ann['category_id']
            if coco_cat_id in cat_id_to_yolo:
                yolo_class_idx = cat_id_to_yolo[coco_cat_id]
                current_image_classes.add(yolo_class_idx)

        for cls_idx in current_image_classes:
            if cls_idx not in classes_found:
                sample_images_per_class[cls_idx] = {
                    'file_name': img_info['file_name'],
                    'image_id': img_id,
                    'width': img_info['width'],
                    'height': img_info['height']
                }
                classes_found.add(cls_idx)

        if len(classes_found) == len(yolo_names): # Found a sample for all classes
            break

    # --- Visualize the sample images with annotations ---
    print(f"\nVisualizing {len(sample_images_per_class)} sample images, one for each class...")
    fig, axes = plt.subplots(3, 4, figsize=(15, 12))
    axes = axes.flatten()

    for i, (cls_idx, img_data) in enumerate(sample_images_per_class.items()):
        ax = axes[i]
        file_name = img_data['file_name']
        image_id = img_data['image_id']
        img_w = img_data['width']
        img_h = img_data['height']

        img_path = IMAGE_DIR_TRAINING / file_name
        if not img_path.exists():
            print(f"Warning: Image not found at {img_path}. Skipping.")
            ax.set_title(f"Missing: {yolo_names[cls_idx]}")
            ax.axis('off')
            continue

        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"Class: {yolo_names[cls_idx]}")
        ax.axis('off')

        # Plot bounding boxes for the chosen image
        if image_id in img_to_anns:
            for ann in img_to_anns[image_id]:
                coco_cat_id = ann['category_id']
                if coco_cat_id in cat_id_to_yolo:
                    # YOLO index and COCO ID are not necessarily the same
                    # Ensure we are only plotting for the current class in focus for better clarity
                    # Or plot all annotations for the image
                    x, y, w, h = ann['bbox'] # COCO format: [x_min, y_min, width, height]
                    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none')
                    ax.add_patch(rect)
                    # Optionally add class name to bbox
                    ax.text(x, y-5, yolo_names[cat_id_to_yolo[coco_cat_id]], color='red', fontsize=8, backgroundcolor='white')

    plt.tight_layout()
    plt.show()
else:
    print("Could not load training data for visualization.")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
!pip install ultralytics transformers peft accelerate

## Using YOLO

### Converting to YOLO Format

In [ ]:
# import ultralytics
# import os
# import json
# import yaml
# from pathlib import Path
# from tqdm import tqdm
# from collections import defaultdict # Added for img_to_anns

# # --- Configuration --- For the YOLO dataset structure
# YOLO_ROOT = RUN_ROOT / 'yolo_dataset'

# TRAIN_IMAGES_OUT = YOLO_ROOT / 'images' / 'train'
# TRAIN_LABELS_OUT = YOLO_ROOT / 'labels' / 'train'

# VAL_IMAGES_OUT = YOLO_ROOT / 'images' / 'val' # For validation data (test data)
# VAL_LABELS_OUT = YOLO_ROOT / 'labels' / 'val'   # For validation data (test data)

# # Ensure root directories exist
# YOLO_ROOT.mkdir(parents=True, exist_ok=True)

# print(f"Setting up YOLO data in: {YOLO_ROOT}")

# def convert_coco_to_yolo(json_path, image_source_dir, images_out_dir, labels_out_dir, cat_id_to_yolo):
#     """
#     Converts COCO annotations to YOLO format and symlinks images.
#     """
#     images_out_dir.mkdir(parents=True, exist_ok=True)
#     labels_out_dir.mkdir(parents=True, exist_ok=True)

#     if not json_path.exists():
#         print(f"Warning: {json_path} not found. Skipping conversion for this path.")
#         return

#     with open(json_path, 'r') as f:
#         coco_data = json.load(f)

#     # Group annotations by image_id
#     img_to_anns = defaultdict(list)
#     for ann in coco_data['annotations']:
#         img_to_anns[ann['image_id']].append(ann)

#     print(f"Converting annotations and symlinking images from {json_path.name} to {images_out_dir}...")
#     for img_info in tqdm(coco_data['images']):
#         img_id = img_info['id']
#         file_name = img_info['file_name']

#         # Source image path
#         src_path = image_source_dir / file_name

#         # 1. Symlink Image (Avoid copying to save space/time)
#         dst_img_path = images_out_dir / file_name
#         if not dst_img_path.exists(): # Only create symlink if it doesn't exist
#             try:
#                 os.symlink(src_path.resolve(), dst_img_path)
#             except Exception as e:
#                 # Fallback to copy if symlink fails (e.g. some drive systems)
#                 import shutil
#                 shutil.copy(src_path, dst_img_path)

#         # 2. Create Label File
#         # YOLO format: class_id x_center y_center width height (normalized 0-1)
#         img_w = img_info['width']
#         img_h = img_info['height']

#         label_file = labels_out_dir / f"{Path(file_name).stem}.txt"

#         anns = img_to_anns.get(img_id, [])
#         yolo_lines = []

#         for ann in anns:
#             cat_id = ann['category_id']
#             if cat_id not in cat_id_to_yolo:
#                 # This should ideally not happen if cat_id_to_yolo is correctly built from all categories
#                 continue

#             cls_idx = cat_id_to_yolo[cat_id]
#             bbox = ann['bbox'] # [x_min, y_min, width, height]

#             # Convert to normalized center-x, center-y, w, h
#             x_min, y_min, w, h = bbox

#             x_center = (x_min + w / 2) / img_w
#             y_center = (y_min + h / 2) / img_h
#             w_norm = w / img_w
#             h_norm = h / img_h

#             # Clip values to [0, 1] just in case of float inaccuracies
#             x_center = max(0.0, min(1.0, x_center))
#             y_center = max(0.0, min(1.0, y_center))
#             w_norm = max(0.0, min(1.0, w_norm))
#             h_norm = max(0.0, min(1.0, h_norm))

#             yolo_lines.append(f"{cls_idx} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

#         # Write to file only if there are annotations
#         if yolo_lines:
#             with open(label_file, 'w') as f:
#                 f.write("\n".join(yolo_lines))
#         else:
#             # Create an empty file if no annotations, YOLO expects a label file for every image
#             open(label_file, 'a').close()

#     print(f"Conversion for {json_path.name} complete.")

# # --- Load categories and create mapping (from training data for consistency) ---
# cat_id_to_yolo = {}
# yolo_names = {}

# if TRAIN_JSON_PATH.exists():
#     with open(TRAIN_JSON_PATH, 'r') as f:
#         train_coco_data = json.load(f)

#     # Map category IDs to continuous YOLO indices (0, 1, 2...) from training data
#     cats = sorted(train_coco_data['categories'], key=lambda x: x['id'])
#     cat_id_to_yolo = {c['id']: i for i, c in enumerate(cats)}
#     yolo_names = {i: c['name'] for i, c in enumerate(cats)}

#     print(f"Classes mapping: {yolo_names}")

#     # --- Convert Training Data ---
#     convert_coco_to_yolo(
#         TRAIN_JSON_PATH,
#         IMAGE_DIR_TRAINING,
#         TRAIN_IMAGES_OUT,
#         TRAIN_LABELS_OUT,
#         cat_id_to_yolo
#     )

#     # --- Convert Testing Data (for validation) ---
#     convert_coco_to_yolo(
#         TEST_JSON_PATH,
#         IMAGE_DIR_TESTING,
#         VAL_IMAGES_OUT,
#         VAL_LABELS_OUT,
#         cat_id_to_yolo
#     )

#     print("\nData conversion complete for both training and testing datasets.")

#     # --- Create data.yaml --- YOLOv11 expects a YAML file pointing to the data
#     yaml_content = {
#         'path': str(YOLO_ROOT.resolve()), # Dataset root dir
#         'train': 'images/train',          # Train images (relative to 'path')
#         'val': 'images/val',              # Validation images (relative to 'path')
#         'nc': len(yolo_names),
#         'names': yolo_names
#     }

#     yaml_path = YOLO_ROOT / 'data.yaml'
#     with open(yaml_path, 'w') as f:
#         yaml.dump(yaml_content, f, sort_keys=False)

#     print(f"Created configuration file at: {yaml_path}")
#     print("Setup ready for YOLOv11 training.")

# else:
#     print("Error: Training JSON file not found. Cannot convert data and create data.yaml.")

In [ ]:
YOLO_ROOT = RUN_ROOT / 'yolo_dataset'
IMAGES_OUT = YOLO_ROOT / 'images' / 'train'

LABELS_OUT = YOLO_ROOT / 'labels' / 'train'

### Training

In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO('yolo11s.pt')

# Define path to data.yaml
# Ensure this matches the path created in the previous cell
yaml_path = YOLO_ROOT / 'data.yaml'

# Train the model
print(f"Starting training on device: {device}")

results = model.train(
    data=str(yaml_path),
    epochs=50, # Number of training epochs
    imgsz=640, # Correct standard size for YOLO detection (vs 224 for classification)
    batch=64, # Batch size
    device=str(device), # Use the device defined earlier (cuda/cpu)
    project=str(RUN_ROOT / 'runs/train'), # Save results directly to Drive
    name='yolo11_xray', # Name of the experiment
    exist_ok=True, # Allow overwriting existing experiment folder
    optimizer="AdamW"
)

Starting training on device: cuda
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/My Drive/Deep_Learning_Project/Training_and_annotations/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11_xray, nbs=64, nms=False, opset=None

### Evaluation

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import random

# Define path to the best model weights
# (Saved automatically by YOLO in the project/name/weights folder)
best_model_path = RUN_ROOT / 'runs/train/yolo11_xray/weights/best.pt'

if best_model_path.exists():
    print(f"Loading best model from: {best_model_path}")
    best_model = YOLO(best_model_path)

    # Run Validation on the dataset
    print("\nRunning validation...")
    metrics = best_model.val(data=str(YOLO_ROOT / 'data.yaml'), device=str(device))
    print(f"mAP50-95: {metrics.box.map}")

    # Run Inference on random samples from the validation set
    print("\nRunning inference on random samples from the validation set...")
    # Get list of validation images
    val_imgs_dir = YOLO_ROOT / 'images' / 'val'
    image_files = list(val_imgs_dir.glob('*.png')) # Adjust extension if needed

    if image_files:
        # Pick 4 random images
        test_images = random.sample(image_files, min(16, len(image_files)))

        # Run prediction
        results = best_model.predict(test_images, device=str(device))

        # Visualize results
        for r in results:
            # Plot returns a BGR numpy array
            im_array = r.plot()

            plt.figure(figsize=(8, 8))
            # Convert BGR to RGB for matplotlib
            plt.imshow(im_array[..., ::-1])
            plt.axis('off')
            plt.title(f"Prediction: {Path(r.path).name}")
            plt.show()
    else:
        print("No images found for inference in the validation set.")
else:
    print(f"Best model not found at {best_model_path}.\nEnsure the training cell above has finished successfully.")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import pandas as pd

# Define path to results file generated during training
# Corrected path to include the 'train' subdirectory
results_path = RUN_ROOT / 'runs/train/yolo11_xray/results.csv'

if results_path.exists():
    # Load data into a pandas DataFrame
    df = pd.read_csv(results_path)

    # Clean column names (remove extra spaces often found in YOLO CSVs)
    df.columns = [c.strip() for c in df.columns]

    # Calculate F1-Score
    # F1 = 2 * (Precision * Recall) / (Precision + Recall)
    # We use the box metrics: metrics/precision(B) and metrics/recall(B)
    # Add epsilon 1e-16 to avoid division by zero
    p = df['metrics/precision(B)']
    r = df['metrics/recall(B)']
    df['metrics/F1'] = 2 * (p * r) / (p + r + 1e-16)

    # Analyze Best Performance
    # Find the row with the best mAP50-95
    best_idx = df['metrics/mAP50-95(B)'].idxmax()
    best_epoch = df.iloc[best_idx]

    map50 = best_epoch['metrics/mAP50(B)']
    map50_95 = best_epoch['metrics/mAP50-95(B)']
    precision = best_epoch['metrics/precision(B)']
    recall = best_epoch['metrics/recall(B)']
    f1 = best_epoch['metrics/F1']

    print("\n--- Model Performance Explanation ---")
    print(f"Peak Performance achieved at Epoch {int(best_epoch['epoch'])}")
    print(f"• mAP@50: {map50:.3%}")
    print(f"• mAP@50-95: {map50_95:.3%}")
    print(f"• Precision: {precision:.3%}")
    print(f"• Recall: {recall:.3%}")
    print(f"• F1-Score: {f1:.3%}")

    print("\nInterpretation:")

    # F1 Interpretation
    if f1 > 0.90:
        print("Excellent Balance: The F1-score indicates a top-tier balance between precision and recall.")
    elif f1 > 0.75:
        print("Good Balance: The model maintains a strong trade-off between catching objects and being correct.")
    else:
        print("Imbalanced: The model might be biased towards either missing objects or generating false alarms.")

    # mAP@50 Interpretation
    if map50 > 0.90:
        print("Excellent Detection Accuracy: The model correctly identifies objects with an overlap of 50% or more over 90% of the time.")
    elif map50 > 0.75:
        print("Good Detection Accuracy: The model is reliable but may occasionally miss difficult or obscured objects.")
    else:
        print("Moderate Detection: The model might struggle with complex scenes. Consider more training epochs.")

    # mAP@50-95 Interpretation
    if map50_95 > 0.70:
        print("High Localization Precision: The bounding boxes are very tight and match the objects' shapes closely.")
    elif map50_95 > 0.50:
        print("Good Localization: The model finds the objects well, though the boxes might be slightly loose.")
    else:
        print("Low Localization: The model finds the objects but the boxes are often inaccurate (too big or offset).")

else:
    print("Results file not found. Please ensure the training cell has completed successfully.")


--- Model Performance Explanation ---
Peak Performance achieved at Epoch 50
• mAP@50: 73.779%
• mAP@50-95: 60.101%
• Precision: 78.361%
• Recall: 67.214%
• F1-Score: 72.361%

Interpretation:
Imbalanced: The model might be biased towards either missing objects or generating false alarms.
Moderate Detection: The model might struggle with complex scenes. Consider more training epochs.
Good Localization: The model finds the objects well, though the boxes might be slightly loose.
